# Inference for the True Slope

Our simulations show that if the regression model holds and the sample size is large, then the regression line is likely to be close to the true line. This allows us to estimate the slope of the true line.

We will use our familiar sample of mothers and their newborn babies to develop a method of estimating the slope of the true line. First, let's see if we believe that the regression model is an appropriate set of assumptions for describing the relation between birth weight and the number of gestational days.

In [1]:
import polars as pl
import numpy as np
import hvplot.polars
import holoviews as hv
from datascience import make_array, percentile
path_data = '../../../assets/data/'
np.set_printoptions(legacy='1.13')

def hist_column(values, xlabel, bins=None):
    """POLARS_CHANGE: hvplot.hist draws a density, matching Table.hist and visualizations.ipynb."""
    frame = pl.DataFrame([pl.Series(xlabel, values)])
    if bins is None:
        bins = 10
    return frame.hvplot.hist(xlabel, bins=bins, normed=True, xlabel=xlabel)

In [2]:
def standard_units(any_numbers):
    "Convert any Series of numbers to standard units."
    return (any_numbers - any_numbers.mean())/any_numbers.std(ddof=0)  

def correlation(t, x, y):
    return (standard_units(t.get_column(x))*standard_units(t.get_column(y))).mean()

def slope(table, x, y):
    r = correlation(table, x, y)
    return r * table.get_column(y).std(ddof=0)/table.get_column(x).std(ddof=0)

def intercept(table, x, y):
    a = slope(table, x, y)
    return table.get_column(y).mean() - a * table.get_column(x).mean()

def fit(table, x, y):
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.get_column(x) + b

def residual(table, x, y):
    return table.get_column(y) - fit(table, x, y)

def scatter_fit(table, x, y):
    """POLARS_CHANGE: hvplot.scatter with the regression line overlaid in yellow with a thin black outline. The chart is returned so a cell can show it or draw on it."""
    points = table.hvplot.scatter(x=x, y=y, color='#002fa7')
    fitted = pl.DataFrame([table.get_column(x), fit(table, x, y).alias('Fitted value')]).sort(x)
    line = (
        fitted.hvplot.line(x=x, y='Fitted value', color='black', line_width=3)  # dark outline for contrast
        * fitted.hvplot.line(x=x, y='Fitted value', color='#FFFF00', line_width=2)
    )
    return (points * line).opts(xlabel=x, ylabel=y)

In [3]:
def draw_and_compare(true_slope, true_int, sample_size):
    x = np.random.normal(50, 5, sample_size)
    xlims = make_array(np.min(x), np.max(x))
    eps = np.random.normal(0, 6, sample_size)
    y = (true_slope*x + true_int) + eps
    tyche = pl.DataFrame([
        pl.Series('x', x),
        pl.Series('y', y),
    ])

    true_line = hv.Curve((xlims, true_slope*xlims + true_int)).opts(color='#1e6400', line_width=2)
    graph_1 = (tyche.hvplot.scatter(x='x', y='y', color='#002fa7') * true_line).opts(title='True Line, and Points Created')

    graph_2 = tyche.hvplot.scatter(x='x', y='y', color='#002fa7').opts(title='What We Get to See')

    graph_3 = scatter_fit(tyche, 'x', 'y').opts(xlabel='', ylabel='', title='Regression Line: Estimate of True Line')

    xlims = make_array(tyche.get_column('x').min(), tyche.get_column('x').max())
    true_line = hv.Curve((xlims, true_slope*xlims + true_int)).opts(color='#1e6400', line_width=2)
    graph_4 = (scatter_fit(tyche, 'x', 'y') * true_line).opts(ylabel='', title='Regression Line and True Line')

    # POLARS_CHANGE: the four matplotlib subplots are a HoloViews layout with one graph per row
    return (graph_1 + graph_2 + graph_3 + graph_4).cols(1)

In [4]:
baby = pl.read_csv(path_data + 'baby.csv')

In [5]:
scatter_fit(baby, 'Gestational Days', 'Birth Weight')

Scatterplot with 'Gestational Days' on the x-axis and 'Birth Weight' on the y-axis. The data points are shown in blue and generally have x values greater than x=225 and a weak to moderate positive correlation. Some outliers exist with low x values, around 150, 180, and just above 200. Each of these outlier data points have y values below 120. A yellow line is shown with a positive slope through the majority of the datapoints.

In [6]:
correlation(baby, 'Gestational Days', 'Birth Weight')

0.40754279338885097

By and large, the scatter looks fairly evenly distributed around the line, though there are some points that are scattered on the outskirts of the main cloud. The correlation is 0.4 and the regression line has a positive slope. 

Does this reflect the fact that the true line has a positive slope? To answer this question, let us see if we can estimate the true slope. We certainly have one estimate of it: the slope of our regression line. That's about 0.47 ounces per day.

In [7]:
slope(baby, 'Gestational Days', 'Birth Weight')

0.4665568769492151

But had the scatter plot come out differently, the regression line would have been different and might have had a different slope. How do we figure out how different the slope might have been?

We need another sample of points, so that we can draw the regression line through the new scatter plot and find its slope. But from where will get another sample?

You have guessed it – we will *bootstrap our original sample*. That will give us a bootstrapped scatter plot, through which we can draw a regression line.

## Bootstrapping the Scatter Plot
We can simulate new samples by random sampling with replacement from the original sample, as many times as the original sample size. Each of these new samples will give us a scatter plot. We will call that a *bootstrapped scatter plot*, and for short, we will call the entire process *bootstrapping the scatter plot*.

Here is the original scatter diagram from the sample, and four replications of the bootstrap resampling procedure. Notice how the resampled scatter plots are in general a little more sparse than the original. That is because some of the original points do not get selected in the samples.

In [8]:
graphs = [baby.hvplot.scatter(x='Gestational Days', y='Birth Weight', color='#002fa7', xlim=(150, 400), title='Original sample')]

for i in np.arange(1, 5, 1):
    rep = baby.sample(n=baby.height, with_replacement=True)
    graphs.append(rep.hvplot.scatter(x='Gestational Days', y='Birth Weight', color='#002fa7', xlim=(150, 400), title='Bootstrap sample '+str(i)))

hv.Layout(graphs).cols(1)

Five scatterplots are shown. The first is titled 'Original sample,' and the remaining are titled 'Bootstrap sample 1' to 'Bootstrap sample 4.' The 'Original sample' is the same data as before. The bootstrap samples resemble the original sample but have slight differences. For example, some have one or two of the outliers, but not all of them.

## Estimating the True Slope

We can bootstrap the scatter plot a large number of times, and draw a regression line through each bootstrapped plot. Each of those lines has a slope. We can simply collect all the slopes and draw their empirical histogram. Each bootstrap sample is drawn at random with replacement, the same number of times as the number of rows in the table.

POLARS_CHANGE: In Data 8, `sample` with no arguments generates a bootstrap sample. Polars `sample` draws one row without replacement unless told otherwise, so a bootstrap sample is `baby.sample(n=baby.height, with_replacement=True)`.

In [9]:
slopes = make_array()
for i in np.arange(5000):
    bootstrap_sample = baby.sample(n=baby.height, with_replacement=True)
    bootstrap_slope = slope(bootstrap_sample, 'Gestational Days', 'Birth Weight')
    slopes = np.append(slopes, bootstrap_slope)
hist_column(slopes, 'Bootstrap Slopes', bins=20)

A histogram with 'Bootstrap Slopes' on the x-axis and 'Percent per unit' on the y-axis. The histogram is symmetric and bell shaped ranging from about 0.35 to 0.6 and centered around 0.475.

We can then construct an approximate 95% confidence interval for the slope of the true line, using the bootstrap percentile method. The confidence interval extends from the 2.5th percentile to the 97.5th percentile of the 5000 bootstrapped slopes.

In [10]:
left = percentile(2.5, slopes)
right = percentile(97.5, slopes)
left, right

(0.38300206648391449, 0.55473291612771292)

An approximate 95% confidence interval for the true slope extends from about 0.38 ounces per day to about 0.55 ounces per day. 

## A Function to Bootstrap the Slope

Let us collect all the steps of our method of estimating the slope and define a function `bootstrap_slope` that carries them out. Its arguments are the name of the table and the labels of the predictor and response variables, and the desired number of bootstrap replications. In each replication, the function bootstraps the original scatter plot and calculates the slope of the resulting regression line. It then draws the histogram of all the generated slopes and prints the interval consisting of the "middle 95%" of the slopes. 

In [11]:
def bootstrap_slope(table, x, y, repetitions):
    
    # For each repetition:
    # Bootstrap the scatter, get the slope of the regression line,
    # augment the list of generated slopes
    slopes = make_array()
    for i in np.arange(repetitions):
        bootstrap_sample = table.sample(n=table.height, with_replacement=True)
        bootstrap_slope = slope(bootstrap_sample, x, y)
        slopes = np.append(slopes, bootstrap_slope)
    
    # Find the endpoints of the 95% confidence interval for the true slope
    left = percentile(2.5, slopes)
    right = percentile(97.5, slopes)
    
    # Slope of the regression line from the original sample
    observed_slope = slope(table, x, y)
    
    # Display results
    chart = hist_column(slopes, 'Bootstrap Slopes', bins=20)
    interval = (
        hv.Curve((make_array(left, right), make_array(0, 0))).opts(color='black', line_width=9)  # dark outline for contrast
        * hv.Curve((make_array(left, right), make_array(0, 0))).opts(color='#FFFF00', line_width=8)
    )
    print('Slope of regression line:', observed_slope)
    print('Approximate 95%-confidence interval for the true slope:')
    print(left, right)
    return chart * interval

When we call `bootstrap_slope` to find a confidence interval for the true slope when the response variable is birth weight and the predictor is gestational days, we get an interval very close to the one we obtained earlier: approximately 0.38 ounces per day to 0.56 ounces per day.

In [12]:
bootstrap_slope(baby, 'Gestational Days', 'Birth Weight', 5000)

Slope of regression line: 0.4665568769492151
Approximate 95%-confidence interval for the true slope:
0.381327939907 0.554839004261


The previous histogram has been reproduced with a yellow bar extending along the bottom of the histogram from about 0.375 to 0.55. The previous histogram description is: A histogram with 'Bootstrap Slopes' on the x-axis and 'Percent per unit' on the y-axis. The histogram is symmetric and bell shaped ranging from about 0.35 to 0.6 and centered around 0.475.

Now that we have a function that automates our process of estimating the slope of the true line in a regression model, we can use it on other variables as well.

For example, let's examine the relation between birth weight and the mother's height. Do taller women tend to have heavier babies?

The regression model seems reasonable, based on the scatter plot, but the correlation is not high. It's just about 0.2.

In [13]:
scatter_fit(baby, 'Maternal Height', 'Birth Weight')

A scatterplot with 'Maternal Height' on the x-axis and 'Birth Weight' on the y-axis. The data points are in light blue and have discrete x values. For a single x-value, there are generally multiple y-values plotted. Most of the data points have x>55. A yellow line is drawn with a slight positive slope.

In [14]:
correlation(baby, 'Maternal Height', 'Birth Weight')

0.20370417718968034

As before, we can use `bootstrap_slope` to estimate the slope of the true line in the regression model.

In [15]:
bootstrap_slope(baby, 'Maternal Height', 'Birth Weight', 5000)

Slope of regression line: 1.478019351928433
Approximate 95%-confidence interval for the true slope:
1.04489613376 1.89613018233


A histogram with 'Boostrap Slopes' on the x-axis and 'Percent per unit' on the y-axis. The histogram is symmetric and bell shaped, extending from about 0.75 to 2.25 and centered around 1.50. A yellow bar is drawn on the bottom of the histogram extending from just above 1.00 to just below 2.00.

A 95% confidence interval for the true slope extends from about 1 ounce per inch to about 1.9 ounces per inch.

## Could the True Slope Be 0?

Suppose we believe that our data follow the regression model, and we fit the regression line to estimate the true line. If the regression line isn't perfectly flat, as is almost invariably the case, we will be observing some linear association in the scatter plot.

But what if that observation is spurious? In other words, what if the true line was flat – that is, there was no linear relation between the two variables – and the association that we observed was just due to randomness in generating the points that form our sample?

Here is a simulation that illustrates why this question arises. We will once again call the function ``draw_and_compare``, this time requiring the true line to have slope 0. Our goal is to see whether our regression line shows a slope that is not 0.

Remember that the arguments to the function ``draw_and_compare`` are the slope and the intercept of the true line, and the number of points to be generated.

In [16]:
draw_and_compare(0, 10, 25)

Four graphs are drawn. The first graph is titled 'True Line, and Points Created,' with a green line drawn at y=10 and light blue data points drawn above and below the line. On the left hand side of the graph there are more data points below the green line than above it and on the right hand side of the graph there are more data points above the green line than below it. The second graph is titled 'What We Get to See' and only has the data points drawn. The third graph is titled 'Regression Line: Estimate of True Line' and has the same data points drawn along with a yellow line with a slight positive slope. The final graph is titled 'Regression Line and True Line' and has the data points and both the yellow and green lines drawn.

Run the simulation a few times, keeping the slope of the true line 0 each time. You will notice that while the slope of the true line is 0, the slope of the regression line is typically not 0. The regression line sometimes slopes upwards, and sometimes downwards, each time giving us a false impression that the two variables are correlated.

To decide whether or not the slope that we are seeing is real, we would like to test the following hypotheses:

**Null Hypothesis.** The slope of the true line is 0. 

**Alternative Hypothesis.** The slope of the true line is not 0. 

We are well positioned to do this. Since we can construct a 95% confidence interval for the true slope, all we have to do is see whether the interval contains 0. 

If it doesn't, then we can reject the null hypothesis (with the 5% cutoff for the P-value). 

If the confidence interval for the true slope does contain 0, then we don't have enough evidence to reject the null hypothesis. Perhaps the slope that we are seeing is spurious.

Let's use this method in an example. Suppose we try to estimate the birth weight of the baby based on the mother's age. Based on the sample, the slope of the regression line for estimating birth weight based on maternal age is positive, about 0.08 ounces per year. 

In [17]:
slope(baby, 'Maternal Age', 'Birth Weight')

0.0850076694158251

Though the slope is positive, it's pretty small. The regression line is so close to flat that it raises the question of whether the true line is flat.

In [18]:
scatter_fit(baby, 'Maternal Age', 'Birth Weight')

A scatterplot with 'Maternal Age' on the x-axis and 'Birth Weight' on the y-axis. The data points are shown in light blue and have discrete x values. There is no apparent shape to the data. A yellow line is drawn with a very slight positive slope.

We can use `bootstrap_slope` to estimate the slope of the true line. The calculation shows that an approximate 95% bootstrap confidence interval for the true slope has a negative left end point and a positive right end point – in other words, the interval contains 0. 

In [19]:
bootstrap_slope(baby, 'Maternal Age', 'Birth Weight', 5000)

Slope of regression line: 0.0850076694158251
Approximate 95%-confidence interval for the true slope:
-0.10377781741 0.270514349998


A histogram with 'Bootstrap Slopes' on the x-axis and 'Percent per unit' on the y-axis. The histogram is centered about 0.1, symmetric, and bell shaped ranging from about -0.2 to 0.35. A yellow bar is shown at the bottom of the histogram ranging from about -0.1 to about 0.27, so it includes 0

Because the interval contains 0, we cannot reject the null hypothesis that the slope of the true linear relation between maternal age and baby's birth weight is 0. Based on this analysis, it would be unwise to predict birth weight based on the regression model with maternal age as the predictor.